## Hey Larry — Wake Word Training

**Instructions:** 1) Runtime → Change runtime type → T4 GPU. 2) Runtime → Run all. 3) Wait ~1–2 hours. 4) Files download automatically at the end (`hey_larry.tflite` + `hey_larry.onnx`).

Training generates synthetic speech via Piper TTS (English, libritts_r-medium voice), augments it with background noise and room impulse responses, and trains a small DNN classifier using the openWakeWord framework. No edits needed — just Run All.

## Training your own openWakeWord models


**Quick-start:** If you just want to train a basic custom model for openWakeWord!

Follow the instructions for Step 1 below. Each time you change the wake word, click the play icon to the left of the title to generate a sample and make sure it sounds correct. The first time it takes a few minutes but subsequent runs will be quick.

Once you're satisfied with the pronounciation, go to the "Runtime" dropdown menu in the upper left of the page, and select "run all". Keep the tab open but feel free to do something else. After ~1 hour, your custom model will be ready and will automatically be downloaded to your computer!

If you are a Home Assistant user with the openWakeWord add-on, follow the instructions [here](https://github.com/home-assistant/addons/blob/master/openwakeword/DOCS.md#custom-wake-word-models) to install and enable your custom model.

---

If you are interested in learning more about the custom model training process (and increasing the accuracy of your custom models), read through each step in this notebook and try experimenting with different training parameters. If you have any questions or problems, feel free to start a discussion at the openWakeWord [repo](https://github.com/dscripka/openWakeWord/discussions).

In [ ]:
# @title  { display-mode: "form" }
# @markdown # 1. Test Example Training Clip Generation
# @markdown Since openWakeWord models are trained on synthetic examples of your
# @markdown target wake word, it's a good idea to make sure that the examples
# @markdown sound correct. Type in your target wake word below, and run the
# @markdown cell to listen to it.
# @markdown
# @markdown Tips for pronunciation:
# @markdown - Spell out phonetically: "hey siri" --> "hey_seer_e"
# @markdown - Spell out numbers: "2" --> "two"
# @markdown - No punctuation except "?" and "!"

import os
import sys
from IPython.display import Audio

if not os.path.exists('./piper-sample-generator'):
    !git clone https://github.com/rhasspy/piper-sample-generator
    # Pin to commit before Mar-2026 repackaging that broke `import generate_samples`
    # Parent of 1a8c49bd29b3a132721086ee88f2253f788594a8 (repackage commit)
    !cd piper-sample-generator && git checkout c9d824c0e2cce8bdeb000c219dc9cbc84df086ea
    os.makedirs('piper-sample-generator/models', exist_ok=True)
    # English voice: libritts_r-medium
    !wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.onnx 'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium/en_US-libritts_r-medium.onnx'
    !wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.onnx.json 'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium/en_US-libritts_r-medium.onnx.json'
    !pip install -q piper-tts piper-phonemize-cross
    !pip install -q webrtcvad
    !pip install -q 'torch<=2.5' torchvision torchaudio

if 'piper-sample-generator/' not in sys.path:
    sys.path.append('piper-sample-generator/')

import generate_samples

target_word = 'Hey_Larry'  # @param {type:"string"}

def text_to_speech(text):
    generate_samples.generate_samples_onnx(
        text=text,
        max_samples=1,
        length_scales=[1.1],
        noise_scales=[0.7], noise_scale_ws=[0.7],
        output_dir='./', batch_size=1, auto_reduce_batch_size=True,
        file_names=['test_generation.wav'],
        model='piper-sample-generator/models/en_US-libritts_r-medium.onnx'
    )

text_to_speech(target_word)
Audio('test_generation.wav', autoplay=True)


In [ ]:
# Patch torch_audiomentations for torchaudio>=2.1 (set_audio_backend was removed).
# Locate the installed file dynamically instead of guessing the path.
import importlib.util, re
spec = importlib.util.find_spec('torch_audiomentations.utils.io')
io_path = spec.origin
s = open(io_path).read()
s2 = re.sub(r'torchaudio\.set_audio_backend\([^)]*\)', 'pass  # removed in torchaudio>=2.1', s)
open(io_path,'w').write(s2)
print('patched:', io_path, '| changed:', s != s2)


In [ ]:
# @title  { display-mode: "form" }
# @markdown # 3. Train the Model
# @markdown Adjust training parameters below (or keep the defaults) and train.
# @markdown With defaults this takes 30–60 min on CPU, ~10–15 min on T4 GPU.
# @markdown The trained model files are automatically downloaded at the end.

# Verify numpy is <2.1 before training starts (belt-and-suspenders)
import numpy as _np_check
from packaging.version import Version
assert Version(_np_check.__version__) < Version('2.1'), \
    f'numpy {_np_check.__version__} >= 2.1 will break numba/training. Run: !pip install -q "numpy<2.1"'
print(f'numpy {_np_check.__version__} OK (< 2.1)')

import yaml, sys
config = yaml.load(open('openwakeword/examples/custom_model.yml', 'r').read(), yaml.Loader)

number_of_examples = 1000  # @param {type:"slider", min:100, max:50000, step:50}
number_of_training_steps = 10000  # @param {type:"slider", min:0, max:50000, step:100}
false_activation_penalty = 1500  # @param {type:"slider", min:100, max:5000, step:50}
config['target_phrase'] = [target_word]
config['model_name'] = config['target_phrase'][0].replace(' ', '_')
config['n_samples'] = number_of_examples
config['n_samples_val'] = max(500, number_of_examples//10)
config['steps'] = number_of_training_steps
config['target_accuracy'] = 0.5
config['target_recall'] = 0.25
config['output_dir'] = './my_custom_model'
config['max_negative_weight'] = false_activation_penalty

config['background_paths'] = ['./audioset_16k', './fma']
config['false_positive_validation_data_path'] = 'validation_set_features.npy'
config['feature_data_files'] = {'ACAV100M_sample': 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy'}

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)

# Generate synthetic positive/negative clips
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

# Augment the generated clips with background noise
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

# Train the model
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

# Convert ONNX → tflite using onnx2tf (works on Python 3.12)
import os
onnx_model_path = f"my_custom_model/{config['model_name']}.onnx"
name1 = f"my_custom_model/{config['model_name']}_float32.tflite"
name2 = f"my_custom_model/{config['model_name']}.tflite"
!onnx2tf -i {onnx_model_path} -o my_custom_model/ -kat onnx____Flatten_0
!mv {name1} {name2}

print(f'Training complete! Model: {config["model_name"]}')
print(f'Files: {onnx_model_path} and {name2}')


In [ ]:
# Download trained model files to your computer
from google.colab import files
import os

onnx_path = f"my_custom_model/{config['model_name']}.onnx"
tflite_path = f"my_custom_model/{config['model_name']}.tflite"

if os.path.exists(onnx_path):
    files.download(onnx_path)
    print(f'Downloaded: {onnx_path}')
else:
    print(f'WARNING: {onnx_path} not found — check training output above')

if os.path.exists(tflite_path):
    files.download(tflite_path)
    print(f'Downloaded: {tflite_path}')
else:
    print(f'WARNING: {tflite_path} not found — check training output above')

print('Done! Upload the .tflite file to your Home Assistant openWakeWord add-on.')
